In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Gold
# Notebook        : Dim_FX_Rate
# Target          : procurement.gold.gold_Dim_Date
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook created the dim_date table in the gold layer.
#
# ==============================================================================
# Business Objective
# ==============================================================================
# Create the Date Dimension table 
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:

# ============================================================
# Dim Date Table
# ============================================================
from pyspark.sql.functions import (
    explode, sequence, to_date, col, year, month, dayofmonth,
    quarter, date_format, dayofweek, weekofyear
)

# Define your date range based on your dataset's actual span
# (check min/max dates across PR, PO, GRN, Invoice, Payment first)
start_date = "2021-01-01"
end_date   = "2026-12-31"

date_df = spark.sql(f"SELECT sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day) as date")
date_df = date_df.withColumn("date", explode(col("date")))

gold_dim_date_df = (
    date_df
        .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
        .withColumn("year", year(col("date")))
        .withColumn("month", month(col("date")))
        .withColumn("month_name", date_format(col("date"), "MMMM"))
        .withColumn("quarter", quarter(col("date")))
        .withColumn("quarter_name", concat(lit("Q"), col("quarter")))
        .withColumn("day", dayofmonth(col("date")))
        .withColumn("day_name", date_format(col("date"), "EEEE"))
        .withColumn("day_of_week", dayofweek(col("date")))
        .withColumn("week_of_year", weekofyear(col("date")))
        .withColumn("is_weekend", dayofweek(col("date")).isin([1, 7]))
        .withColumn("fiscal_year", 
            when(col("month") >= 4, col("year") + 1).otherwise(col("year"))  # if FY starts April, adjust as needed
        )
)

write_delta(df=gold_dim_date_df, table_name=GOLD_DIM_DATE)
# display(gold_dim_date_df)